### Import libraries.

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
from bs4 import BeautifulSoup
import pandas as pd
import os
import requests
import json
import csv
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Implemented file handling functions.

In [2]:
def download_image(source, filename):
    with open(filename, 'wb') as f:
        image_content = requests.get(source).content
        f.write(image_content)
        print('Downloaded image at:', filename)

def write_txt(filename, data):
    with open(filename, 'w', encoding='utf-8') as f:
        f.writelines(data)
    return True


def read_txt(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        data = f.readlines()
    return data


def write_json(filename, data):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    return True


def read_json(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data


def write_csv(filename, data):
    with open(filename, 'w', encoding='utf-8', newline='') as f:
        csv_writer = csv.writer(f)
        for row in data:
            csv_writer.writerow(row)
    return True


def read_csv(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        title = next(reader)
        data = [row for row in reader]
    return title, data

### Filter pages that contain reviews.

In [7]:
driver = webdriver.Chrome()
driver.maximize_window()
driver.get(url='https://www.coursera.org/learn/what-is-datascience')
time.sleep(2)

list_cats = []
tag_review = driver.find_elements(by=By.XPATH, value='//*[@id="rendered-content"]/div/main/section[2]/div/div/div[1]/div[3]/div/nav/ul/li/a')
for li in tag_review:
    list_cats.append(li.text)
print(list_cats)

['About', 'Outcomes', 'Modules', 'Recommendations', 'Testimonials', 'Reviews']


##### The ul tag with class 'css-10v4d2r' contains li tags with content from list_cats as shown in the previous cell. Any page that has an li tag with the text 'Reviews' will contain a Review section -> Iterate through each webpage in the list of links collected in Part I and extract the links that satisfy this condition.

In [ ]:
# Read the list of links from file
list_links = read_txt('list_coursera_links.txt')

# Initialize Chrome browser
driver = webdriver.Chrome()
driver.maximize_window()

list_reviews = []

# Iterate through each link in the list
for i in range(0, len(list_links)):
    link = list_links[i].rstrip('\n')
    driver.get(url=link)
    time.sleep(2)

    list_cats = []
    tag_review = driver.find_elements(
        by=By.XPATH,
        value='//*[@id="rendered-content"]/div/main/section[2]/div/div/div[1]/div[3]/div/nav/ul/li/a'
    )
    for li in tag_review:
        list_cats.append(li.text)
    
    if 'Reviews' in list_cats:
        new_link = link.strip('\n') + '/reviews\n'
        list_reviews.append(new_link)

# Close the browser
driver.close()

# Write the list of review links to file
write_txt('list_review_links.txt', list_reviews)
print('Saved successfully!!!')
print(f'Total filtered links: {len(list_reviews)}')

Đã lưu thành công!!!
Có tổng cộng 548 link đã được lọc.


### Scrape data from the filtered links, specifically 548 links.

In [ ]:
list_reviews = read_txt('list_review_links.txt')
driver = webdriver.Chrome()
driver.maximize_window()

for i in range(37, len(list_reviews)):
    

    link = list_reviews[i]
    page_link = link.rstrip('\n')

    driver.get(url=page_link)
    time.sleep(2)

    # Get the title because it contains the course name and number of pages
    title = driver.find_element(by=By.CSS_SELECTOR, value='h2.cds-119.m-y-2.text-secondary.css-1diqjn6.cds-121').text
    # Get total number of reviews
    rvs = int(title.split(' ')[4].replace(',', ''))
    # Get number of pages and course name
    # Since each page contains up to 25 reviews, calculate total pages by dividing total reviews by 25
    # If divisible by 25, keep the result; otherwise, add 1 page
    page = rvs // 25
    coursera_name = title.split(' Reviews for ')[1]
    # Update total number of pages based on condition
    if rvs % 25 == 0:
        page = page
    else:
        page +=1
        
    reviewer_names = []
    dates = []
    review_contents = []
    stars = []
    
    
    for j in range(1, page+1):
        
        page_url = f"{page_link}?page={j}"
        driver.get(url=page_url)
        time.sleep(2)
        
        # Reviewer names
        tag_reviewer_names = driver.find_elements(by=By.CSS_SELECTOR, value='p.reviewerName.p-x-1s.css-vac8rf')
        for tag_reviewer_name in tag_reviewer_names:
            reviewer_name = tag_reviewer_name.find_element(by=By.CSS_SELECTOR, value='span').text
            reviewer_names.append(reviewer_name)

        # Review dates
        tag_date_reviews = driver.find_elements(by=By.CSS_SELECTOR, value='p.dateOfReview.p-x-1s.css-vac8rf')
        for tag_date_review in tag_date_reviews:
            date = tag_date_review.text
            dates.append(date)

        # Review content  
        tag_review_contents = driver.find_elements(By.XPATH, '//*[@id="rendered-content"]/div/div/div[1]/div/div[2]/div/div/div[2]/div/div')
        for tag_review_content in tag_review_contents:
            review_content = []
            
            # Find all <p> tags inside tag_review_content
            paragraphs = tag_review_content.find_elements(By.TAG_NAME, 'p')
            
            # Loop through each <p> tag and extract text
            for paragraph in paragraphs:
                paragraph_text = paragraph.text.strip()  # Get text and remove extra whitespace
                review_content.append(paragraph_text)
            
            # Add review_content to review_contents
            review_contents.append(" ".join(review_content))

        # Rating stars
        tag_stars = driver.find_elements(By.XPATH, value='//*[@id="rendered-content"]/div/div/div[1]/div/div[2]/div/div[1]/div[1]/div')
        for tag_star in tag_stars:
            stars_ = tag_star.find_elements(By.CSS_SELECTOR, value='span._13xsef79.d-inline-block')
            
            filled_stars = 0
            for star in stars_:
                # Find svg tag inside span
                svg = star.find_element(By.CSS_SELECTOR, value='svg._ufjrdd')
                # Find title tag inside svg
                title = svg.find_element(By.CSS_SELECTOR, 'title')
                # Get text content of title
                title_text = title.text
                
                # Check if the title content is 'Filled Star'
                if title_text == 'Filled Star':
                    filled_stars += 1
            
            stars.append(filled_stars)
        # print(len(reviewer_names),
        #     len(dates), len(review_contents),
        #     len(stars) 
    else:
        df_products = pd.DataFrame({
        'coursera name' : coursera_name,
        'review name' : reviewer_names,
        'date of review' :  dates,
        'review content' : review_contents,
        'rating star' : stars
        })
        df_products.to_csv(f'data/reviews/{i+1}.csv', index=False)
        print('--> CSV file saved')
        print(f'>>> Completed processing website number {i+1}')
else:
    print('Completed!!!')
    driver.close()

--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 38
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 39
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 40
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 41
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 42
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 43
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 44
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 45
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 46
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 47
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 48
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 49
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 50
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 51
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 52
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 53
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 54
--> Đã lưu tập tin csv
>>> Hoàn thành trang web thứ 55
--> Đã lưu

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=126.0.6478.63)
Stacktrace:
	GetHandleVerifier [0x00007FF60C223E42+31618]
	(No symbol) [0x00007FF60C19B0A9]
	(No symbol) [0x00007FF60C05888A]
	(No symbol) [0x00007FF60C02DAE5]
	(No symbol) [0x00007FF60C0D45A7]
	(No symbol) [0x00007FF60C0EC131]
	(No symbol) [0x00007FF60C0CCEB3]
	(No symbol) [0x00007FF60C09A46B]
	(No symbol) [0x00007FF60C09B001]
	GetHandleVerifier [0x00007FF60C52A01D+3202397]
	GetHandleVerifier [0x00007FF60C576A3D+3516285]
	GetHandleVerifier [0x00007FF60C56C4B0+3473904]
	GetHandleVerifier [0x00007FF60C2D5D46+760454]
	(No symbol) [0x00007FF60C1A6B4F]
	(No symbol) [0x00007FF60C1A1CE4]
	(No symbol) [0x00007FF60C1A1E72]
	(No symbol) [0x00007FF60C19121F]
	BaseThreadInitThunk [0x00007FFA3F3F257D+29]
	RtlUserThreadStart [0x00007FFA401EAF28+40]


### Merge CSV files and combine them into a single CSV file.

* Use the learned functions to save the result as review_full.csv.

In [ ]:
coursera_name = []
reviewer_names = []
dates = []
review_contents = []
stars = []

review_csv_folder = 'data/reviews'

for file in os.listdir(review_csv_folder):
    csv_path = os.path.join(review_csv_folder, file)
    
    try:
        df_review = pd.read_csv(csv_path)
        
        # Extend the lists with the contents of the current DataFrame
        coursera_name.extend(df_review['coursera name'].tolist())
        reviewer_names.extend(df_review['review name'].tolist())
        dates.extend(df_review['date of review'].tolist())
        review_contents.extend(df_review['review content'].tolist())
        stars.extend(df_review['rating star'].tolist())
    
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")

df_couseras = pd.DataFrame({
    'Coursera Name': coursera_name,
    'Reviewer Name': reviewer_names,
    'Date of Review': dates,
    'Review Contents': review_contents,
    'Rating star': stars,
})

# Save the combined DataFrame to a CSV file
df_couseras.to_csv('review_full.csv', index=False)

print('Completed !!!')

Đã hoàn thành !!!


#### Since the list contains 548 links, and each link has approximately 2000–3000 rows of data on average, the final result is only partial. A total of 108 links were successfully scraped, yielding 140,804 rows of data.

In [12]:
df = pd.read_csv('review_full.csv')
df

,Coursera Name,Reviewer Name,Date of Review,Review Contents,Rating star
0,What is Data Science?,By Kevin M,"Mar 25, 2019","Very basic course, to be honest. The grading ...",2
1,What is Data Science?,By Jianfei Z,"Jan 10, 2019",Too much introduction. There is no need to spe...,1
2,What is Data Science?,By Aman J,"Jan 6, 2019",Videos often have disjointed or unrelated elem...,2
3,What is Data Science?,By pooja s,"Apr 20, 2019",Too basic. If you live under a rock and have n...,1
4,What is Data Science?,By Lauren J,"Mar 23, 2019","This was a good introductory course, especiall...",5
...,...,...,...,...,...
140799,Machine Learning with Python,By Shreya B,"May 7, 2022",I WANT A CERTIFICATE FOR THIS COURSE,1
140800,Machine Learning with Python,By lauren p,"Aug 29, 2019",Final project is way too vague,1
140801,Machine Learning with Python,By Xe B,"Jun 6, 2023",sub-par course,1
140802,Machine Learning with Python,By Anmol N,"Jul 3, 2022",scamed,1
